In [0]:
from pathlib import Path

test_batch = Path("/Volumes/workspace/migration_dev/landing/batch_002")
if test_batch.exists():
    # Remove files individually (shutil.rmtree is blocked on /Volumes).
    for f in test_batch.rglob("*"):
        if f.is_file():
            f.unlink()
    # Remove empty directories (deepest first).
    for d in sorted(test_batch.rglob("*"), reverse=True):
        if d.is_dir():
            d.rmdir()
    test_batch.rmdir()
    print("batch_002 removed")
else:
    print("batch_002 does not exist")

In [0]:
import csv
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

landing = Path("/Volumes/workspace/migration_dev/landing")
source_batch = landing / "batch_001"
test_batch = landing / "batch_002"

# Prevent accidental replacement of an existing test batch.
if test_batch.exists():
    raise RuntimeError("batch_002 already exists; stop and report this")

with (source_batch / "metadata/statements.csv").open(
    newline="", encoding="utf-8"
) as source:
    reader = csv.DictReader(source)
    columns = reader.fieldnames
    records = list(reader)

if len(records) != 100:
    raise RuntimeError("Expected 100 source metadata records")

(test_batch / "pdfs").mkdir(parents=True)
(test_batch / "metadata").mkdir()

# Copy the synthetic PDFs and adapt their CSV metadata.
for row in records:
    filename = row["file_name"]

    if not filename or Path(filename).name != filename:
        raise RuntimeError("Unsafe filename")

    content = (source_batch / "pdfs" / filename).read_bytes()
    destination = test_batch / "pdfs" / filename
    destination.write_bytes(content)

    row["batch_id"] = "batch_002"
    row["pdf_path"] = str(destination)
    row["source_sha256"] = hashlib.sha256(content).hexdigest()

csv_path = test_batch / "metadata/statements.csv"

with csv_path.open("w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=columns)
    writer.writeheader()
    writer.writerows(records)

manifest = {
    "batch_id": "batch_002",
    "schema_version": 1,
    "status": "READY",
    "expected_document_count": 100,
    "expected_metadata_count": 100,
    "metadata_path": str(csv_path),
    "metadata_sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

(test_batch / "manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

# Simulate a PDF changing after source checksums were supplied.
changed_pdf = test_batch / "pdfs" / records[0]["file_name"]

changed_pdf.write_bytes(
    changed_pdf.read_bytes()
    + b"\n% Deliberate checksum failure for testing\n"
)

actual_hash = hashlib.sha256(changed_pdf.read_bytes()).hexdigest()
assert actual_hash != records[0]["source_sha256"]

print("Test batch: batch_002")
print(f"PDF count: {len(list((test_batch / 'pdfs').glob('*.pdf')))}")
print(f"Metadata count: {len(records)}")
print(f"Changed PDF: {changed_pdf.name}")
print("Checksum mismatch prepared.")